In [11]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder


In [12]:
mnist = fetch_openml('mnist_784', version=1)

X = mnist.data.values / 255.0
y = mnist.target.astype(int).values.reshape(-1, 1)

encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
def relu(x):
    return np.maximum(0, x)


def relu_derivative(x):
    return x > 0


def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

In [14]:
input_size = 784
hidden1 = 128
hidden2 = 64
output_size = 10

In [15]:
W1 = np.random.randn(input_size, hidden1) * np.sqrt(2 / input_size)
b1 = np.zeros((1, hidden1))

W2 = np.random.randn(hidden1, hidden2) * np.sqrt(2 / hidden1)
b2 = np.zeros((1, hidden2))

W3 = np.random.randn(hidden2, output_size) * np.sqrt(2 / hidden2)
b3 = np.zeros((1, output_size))

learning_rate = 0.01
epochs = 40
batch_size = 64

In [16]:
m = X_train.shape[0]

for epoch in range(epochs):
    
    indices = np.random.permutation(m)
    X_train = X_train[indices]
    y_train = y_train[indices]

    for i in range(0, m, batch_size):
        X_batch = X_train[i:i+batch_size]
        y_batch = y_train[i:i+batch_size]

        batch_m = X_batch.shape[0]

        Z1 = np.dot(X_batch, W1) + b1
        A1 = relu(Z1)

        Z2 = np.dot(A1, W2) + b2
        A2 = relu(Z2)

        Z3 = np.dot(A2, W3) + b3
        A3 = softmax(Z3)

        
        loss = -np.mean(np.sum(y_batch * np.log(A3 + 1e-8), axis=1))

        dZ3 = A3 - y_batch
        dW3 = np.dot(A2.T, dZ3) / batch_m
        db3 = np.sum(dZ3, axis=0, keepdims=True) / batch_m

        dA2 = np.dot(dZ3, W3.T)
        dZ2 = dA2 * relu_derivative(Z2)
        dW2 = np.dot(A1.T, dZ2) / batch_m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / batch_m

        dA1 = np.dot(dZ2, W2.T)
        dZ1 = dA1 * relu_derivative(Z1)
        dW1 = np.dot(X_batch.T, dZ1) / batch_m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / batch_m

        W3 -= learning_rate * dW3
        b3 -= learning_rate * db3

        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

    
    Z1 = np.dot(X_train, W1) + b1
    A1 = relu(Z1)
    Z2 = np.dot(A1, W2) + b2
    A2 = relu(Z2)
    Z3 = np.dot(A2, W3) + b3
    A3 = softmax(Z3)

    train_preds = np.argmax(A3, axis=1)
    train_true = np.argmax(y_train, axis=1)
    train_acc = np.mean(train_preds == train_true)

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}")

Epoch 1/40, Loss: 0.3444, Train Acc: 0.8898
Epoch 2/40, Loss: 0.2273, Train Acc: 0.9125
Epoch 3/40, Loss: 0.1657, Train Acc: 0.9230
Epoch 4/40, Loss: 0.4477, Train Acc: 0.9318
Epoch 5/40, Loss: 0.1315, Train Acc: 0.9366
Epoch 6/40, Loss: 0.1980, Train Acc: 0.9424
Epoch 7/40, Loss: 0.1343, Train Acc: 0.9471
Epoch 8/40, Loss: 0.3490, Train Acc: 0.9503
Epoch 9/40, Loss: 0.2004, Train Acc: 0.9523
Epoch 10/40, Loss: 0.0817, Train Acc: 0.9551
Epoch 11/40, Loss: 0.1407, Train Acc: 0.9579
Epoch 12/40, Loss: 0.1119, Train Acc: 0.9603
Epoch 13/40, Loss: 0.2341, Train Acc: 0.9619
Epoch 14/40, Loss: 0.0308, Train Acc: 0.9636
Epoch 15/40, Loss: 0.1732, Train Acc: 0.9650
Epoch 16/40, Loss: 0.3968, Train Acc: 0.9674
Epoch 17/40, Loss: 0.2192, Train Acc: 0.9679
Epoch 18/40, Loss: 0.0866, Train Acc: 0.9700
Epoch 19/40, Loss: 0.1860, Train Acc: 0.9705
Epoch 20/40, Loss: 0.0851, Train Acc: 0.9725
Epoch 21/40, Loss: 0.2797, Train Acc: 0.9731
Epoch 22/40, Loss: 0.0224, Train Acc: 0.9739
Epoch 23/40, Loss: 

In [17]:
Z1 = np.dot(X_test, W1) + b1
A1 = relu(Z1)

Z2 = np.dot(A1, W2) + b2
A2 = relu(Z2)

Z3 = np.dot(A2, W3) + b3
A3 = softmax(Z3)

predictions = np.argmax(A3, axis=1)
true_labels = np.argmax(y_test, axis=1)

accuracy = np.mean(predictions == true_labels)

print("\nTest Accuracy:", accuracy)



Test Accuracy: 0.9686428571428571
